In [19]:
#pip install pm4py
import os
os.getcwd()

'C:\\Users\\obami\\Documents\\Python_Pra\\Thesis_PPM_2024-25'

In [3]:
#import and preprocess data
import numpy as np
import pandas as pd
import pm4py

#Enode Prefix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from keras.preprocessing.sequence import pad_sequences

HelpDesk Log

1. Load data and keep necessary columns

In [11]:
log = pm4py.read_xes("helpdesk.xes") #read in log data
logdata = pm4py.convert_to_dataframe(log) #convert your log to dataframe
logdata.head()

parsing log, completed traces ::   0%|          | 0/4580 [00:00<?, ?it/s]

,concept:name,lifecycle:transition,org:resource,time:timestamp,Activity,Resource,case:concept:name,case:variant,case:variant-index,case:creator
0,Assign seriousness,complete,Value 1,2012-10-09 14:50:17+00:00,Assign seriousness,Value 1,Case1,Variant 12,12,Fluxicon Disco
1,Take in charge ticket,complete,Value 1,2012-10-09 14:51:01+00:00,Take in charge ticket,Value 1,Case1,Variant 12,12,Fluxicon Disco
2,Take in charge ticket,complete,Value 2,2012-10-12 15:02:56+00:00,Take in charge ticket,Value 2,Case1,Variant 12,12,Fluxicon Disco
3,Resolve ticket,complete,Value 1,2012-10-25 11:54:26+00:00,Resolve ticket,Value 1,Case1,Variant 12,12,Fluxicon Disco
4,Closed,complete,Value 3,2012-11-09 12:54:39+00:00,Closed,Value 3,Case1,Variant 12,12,Fluxicon Disco


In [12]:
logdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21348 entries, 0 to 21347
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   concept:name          21348 non-null  object             
 1   lifecycle:transition  21348 non-null  object             
 2   org:resource          21348 non-null  object             
 3   time:timestamp        21348 non-null  datetime64[ns, UTC]
 4   Activity              21348 non-null  object             
 5   Resource              21348 non-null  object             
 6   case:concept:name     21348 non-null  object             
 7   case:variant          21348 non-null  object             
 8   case:variant-index    21348 non-null  int64              
 9   case:creator          21348 non-null  object             
dtypes: datetime64[ns, UTC](1), int64(1), object(8)
memory usage: 1.6+ MB


In [13]:
#extract the columns, sort by time, convert activity to lower case
logdata.rename(columns={"concept:name":"activity", "time:timestamp":"timestamp","case:concept:name":"case_id","case:variant":"variant","org:resource":"resource"},inplace=True)
logdata = logdata.sort_values(by = ["timestamp","case_id"])
df = logdata[["timestamp","activity","case_id"]].copy()
df["activity"] = df["activity"].str.lower()
df.head()

,timestamp,activity,case_id
16857,2010-01-13 08:40:25+00:00,assign seriousness,Case3608
12863,2010-01-13 12:26:04+00:00,assign seriousness,Case2748
19959,2010-01-13 12:30:37+00:00,assign seriousness,Case4284
7168,2010-01-13 13:09:31+00:00,assign seriousness,Case1534
1864,2010-01-13 17:25:25+00:00,assign seriousness,Case406


In [14]:
#Check if there are duplicates
print(df.duplicated().any())
print(df.duplicated().sum())

True
127


In [15]:
#Check activities for a case
df.drop_duplicates(inplace=True)
df.duplicated().sum()

0

In [16]:
df = df.reset_index(drop=True)
df.head()

,timestamp,activity,case_id
0,2010-01-13 08:40:25+00:00,assign seriousness,Case3608
1,2010-01-13 12:26:04+00:00,assign seriousness,Case2748
2,2010-01-13 12:30:37+00:00,assign seriousness,Case4284
3,2010-01-13 13:09:31+00:00,assign seriousness,Case1534
4,2010-01-13 17:25:25+00:00,assign seriousness,Case406


In [17]:
#timestamps are not unique
df["timestamp"].duplicated().any()

True

In [18]:
df.to_csv("helpdesk_df.csv", index = False)

Prepaid Travel Cost

1. Load data and keep necessary columns

In [3]:
log = pm4py.read_xes("PrepaidTravelCost.xes") #read in log data
logdata = pm4py.convert_to_dataframe(log) #convert your log to dataframe
logdata.head()

parsing log, completed traces ::   0%|          | 0/2099 [00:00<?, ?it/s]

,id,org:resource,concept:name,time:timestamp,org:role,case:Rfp_id,case:Permit travel permit number,case:Task,case:OrganizationalEntity,case:RequestedAmount,...,case:Permit BudgetNumber,case:Permit ProjectNumber,case:Project,case:concept:name,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:Cost Type,case:Permit id,case:Permit ActivityNumber,case:RfpNumber
0,st_step 73555_0,STAFF MEMBER,Permit SUBMITTED by EMPLOYEE,2017-01-09 14:48:43+00:00,EMPLOYEE,request for payment 73550,UNKNOWN,task 71977,organizational unit 65463,854.579838,...,budget 6198,UNKNOWN,project 503,request for payment 73550,organizational unit 65455,1979.272104,0,travel permit 73549,UNKNOWN,request for payment number 73551
1,st_step 73554_0,STAFF MEMBER,Permit FINAL_APPROVED by SUPERVISOR,2017-01-09 14:48:55+00:00,SUPERVISOR,request for payment 73550,UNKNOWN,task 71977,organizational unit 65463,854.579838,...,budget 6198,UNKNOWN,project 503,request for payment 73550,organizational unit 65455,1979.272104,0,travel permit 73549,UNKNOWN,request for payment number 73551
2,st_step 73558_0,STAFF MEMBER,Request For Payment SUBMITTED by EMPLOYEE,2017-01-12 11:40:27+00:00,EMPLOYEE,request for payment 73550,UNKNOWN,task 71977,organizational unit 65463,854.579838,...,budget 6198,UNKNOWN,project 503,request for payment 73550,organizational unit 65455,1979.272104,0,travel permit 73549,UNKNOWN,request for payment number 73551
3,st_step 73559_0,STAFF MEMBER,Request For Payment FINAL_APPROVED by SUPERVISOR,2017-01-12 11:41:59+00:00,SUPERVISOR,request for payment 73550,UNKNOWN,task 71977,organizational unit 65463,854.579838,...,budget 6198,UNKNOWN,project 503,request for payment 73550,organizational unit 65455,1979.272104,0,travel permit 73549,UNKNOWN,request for payment number 73551
4,st_step 73557_0,STAFF MEMBER,Request For Payment REJECTED by MISSING,2017-01-12 11:53:07+00:00,MISSING,request for payment 73550,UNKNOWN,task 71977,organizational unit 65463,854.579838,...,budget 6198,UNKNOWN,project 503,request for payment 73550,organizational unit 65455,1979.272104,0,travel permit 73549,UNKNOWN,request for payment number 73551


In [4]:
logdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18246 entries, 0 to 18245
Data columns (total 22 columns):
 #   Column                            Non-Null Count  Dtype              
---  ------                            --------------  -----              
 0   id                                18246 non-null  object             
 1   org:resource                      18246 non-null  object             
 2   concept:name                      18246 non-null  object             
 3   time:timestamp                    18246 non-null  datetime64[ns, UTC]
 4   org:role                          18246 non-null  object             
 5   case:Rfp_id                       18246 non-null  object             
 6   case:Permit travel permit number  18246 non-null  object             
 7   case:Task                         18246 non-null  object             
 8   case:OrganizationalEntity         18246 non-null  object             
 9   case:RequestedAmount              18246 non-null  float64    

In [5]:
logdata["org:resource"].unique()

array(['STAFF MEMBER', 'SYSTEM'], dtype=object)

In [6]:
logdata["concept:name"].unique()

array(['Permit SUBMITTED by EMPLOYEE',
       'Permit FINAL_APPROVED by SUPERVISOR',
       'Request For Payment SUBMITTED by EMPLOYEE',
       'Request For Payment FINAL_APPROVED by SUPERVISOR',
       'Request For Payment REJECTED by MISSING',
       'Permit REJECTED by MISSING', 'Request Payment', 'Payment Handled',
       'Permit APPROVED by SUPERVISOR',
       'Permit FINAL_APPROVED by DIRECTOR',
       'Request For Payment APPROVED by PRE_APPROVER',
       'Permit APPROVED by PRE_APPROVER',
       'Request For Payment REJECTED by PRE_APPROVER',
       'Request For Payment REJECTED by EMPLOYEE',
       'Request For Payment APPROVED by SUPERVISOR',
       'Request For Payment FINAL_APPROVED by DIRECTOR',
       'Permit REJECTED by PRE_APPROVER', 'Permit REJECTED by EMPLOYEE',
       'Request For Payment REJECTED by SUPERVISOR',
       'Permit REJECTED by SUPERVISOR',
       'Request For Payment SAVED by EMPLOYEE',
       'Request For Payment APPROVED by ADMINISTRATION',
       'Req

In [9]:
#logdata["lifecycle:transition"].unique()

In [ ]:
#case:concept:name is caseid
logdata[logdata["case:concept:name"] == "173688"].shape

In [8]:
#extract the columns, sort by time, convert activity to lower case
logdata.rename(columns={"concept:name":"activity", "time:timestamp":"timestamp","case:concept:name":"case_id"},inplace=True)
logdata = logdata.sort_values(by = ["timestamp","case_id"])
df = logdata[["timestamp","activity","case_id"]].copy()
df["activity"] = df["activity"].str.lower()
df.head()

,timestamp,activity,case_id
0,2017-01-09 14:48:43+00:00,permit submitted by employee,request for payment 73550
6,2017-01-09 14:48:43+00:00,permit submitted by employee,request for payment 73552
1,2017-01-09 14:48:55+00:00,permit final_approved by supervisor,request for payment 73550
7,2017-01-09 14:48:55+00:00,permit final_approved by supervisor,request for payment 73552
13,2017-01-10 11:19:16+00:00,permit submitted by employee,request for payment 76316


In [10]:
df["timestamp"] = df["timestamp"].dt.tz_localize(None) 

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18246 entries, 0 to 10124
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  18246 non-null  datetime64[ns]
 1   activity   18246 non-null  object        
 2   case_id    18246 non-null  object        
dtypes: datetime64[ns](1), object(2)
memory usage: 570.2+ KB


In [12]:
#Check if there are duplicates
print(df.duplicated().any())
print(df.duplicated().sum())

False
0


In [13]:
#Check activities for a case
df.drop_duplicates(inplace=True)
df.duplicated().sum()

0

In [14]:
df = df.reset_index(drop=True)
df.head()

,timestamp,activity,case_id
0,2017-01-09 14:48:43,permit submitted by employee,request for payment 73550
1,2017-01-09 14:48:43,permit submitted by employee,request for payment 73552
2,2017-01-09 14:48:55,permit final_approved by supervisor,request for payment 73550
3,2017-01-09 14:48:55,permit final_approved by supervisor,request for payment 73552
4,2017-01-10 11:19:16,permit submitted by employee,request for payment 76316


In [15]:
#timestamps are not unique
df["timestamp"].duplicated().any()

True

In [16]:
df.to_csv("ptc_df.csv", index = False)

Domestic Declarations

1. Load data and keep necessary columns

In [24]:
log = pm4py.read_xes("DomesticDeclarations.xes") #read in log data
logdata = pm4py.convert_to_dataframe(log) #convert your log to dataframe
logdata.head()

parsing log, completed traces ::   0%|          | 0/10500 [00:00<?, ?it/s]

,id,org:resource,concept:name,time:timestamp,org:role,case:id,case:concept:name,case:BudgetNumber,case:DeclarationNumber,case:Amount
0,st_step 86794_0,STAFF MEMBER,Declaration SUBMITTED by EMPLOYEE,2017-01-09 09:49:50+00:00,EMPLOYEE,declaration 86791,declaration 86791,budget 86566,declaration number 86792,26.851205
1,st_step 86793_0,STAFF MEMBER,Declaration FINAL_APPROVED by SUPERVISOR,2017-01-09 11:27:48+00:00,SUPERVISOR,declaration 86791,declaration 86791,budget 86566,declaration number 86792,26.851205
2,dd_declaration 86791_19,SYSTEM,Request Payment,2017-01-10 09:34:44+00:00,UNDEFINED,declaration 86791,declaration 86791,budget 86566,declaration number 86792,26.851205
3,dd_declaration 86791_20,SYSTEM,Payment Handled,2017-01-12 17:31:22+00:00,UNDEFINED,declaration 86791,declaration 86791,budget 86566,declaration number 86792,26.851205
4,st_step 86798_0,STAFF MEMBER,Declaration SUBMITTED by EMPLOYEE,2017-01-09 10:26:14+00:00,EMPLOYEE,declaration 86795,declaration 86795,budget 86566,declaration number 86796,182.464172


In [25]:
logdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56437 entries, 0 to 56436
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   id                      56437 non-null  object             
 1   org:resource            56437 non-null  object             
 2   concept:name            56437 non-null  object             
 3   time:timestamp          56437 non-null  datetime64[ns, UTC]
 4   org:role                56437 non-null  object             
 5   case:id                 56437 non-null  object             
 6   case:concept:name       56437 non-null  object             
 7   case:BudgetNumber       56437 non-null  object             
 8   case:DeclarationNumber  56437 non-null  object             
 9   case:Amount             56437 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(1), object(8)
memory usage: 4.3+ MB


In [26]:
logdata.head()

,id,org:resource,concept:name,time:timestamp,org:role,case:id,case:concept:name,case:BudgetNumber,case:DeclarationNumber,case:Amount
0,st_step 86794_0,STAFF MEMBER,Declaration SUBMITTED by EMPLOYEE,2017-01-09 09:49:50+00:00,EMPLOYEE,declaration 86791,declaration 86791,budget 86566,declaration number 86792,26.851205
1,st_step 86793_0,STAFF MEMBER,Declaration FINAL_APPROVED by SUPERVISOR,2017-01-09 11:27:48+00:00,SUPERVISOR,declaration 86791,declaration 86791,budget 86566,declaration number 86792,26.851205
2,dd_declaration 86791_19,SYSTEM,Request Payment,2017-01-10 09:34:44+00:00,UNDEFINED,declaration 86791,declaration 86791,budget 86566,declaration number 86792,26.851205
3,dd_declaration 86791_20,SYSTEM,Payment Handled,2017-01-12 17:31:22+00:00,UNDEFINED,declaration 86791,declaration 86791,budget 86566,declaration number 86792,26.851205
4,st_step 86798_0,STAFF MEMBER,Declaration SUBMITTED by EMPLOYEE,2017-01-09 10:26:14+00:00,EMPLOYEE,declaration 86795,declaration 86795,budget 86566,declaration number 86796,182.464172


In [27]:
logdata["concept:name"].unique()

array(['Declaration SUBMITTED by EMPLOYEE',
       'Declaration FINAL_APPROVED by SUPERVISOR', 'Request Payment',
       'Payment Handled', 'Declaration APPROVED by PRE_APPROVER',
       'Declaration REJECTED by MISSING',
       'Declaration REJECTED by PRE_APPROVER',
       'Declaration REJECTED by EMPLOYEE',
       'Declaration SAVED by EMPLOYEE',
       'Declaration REJECTED by SUPERVISOR',
       'Declaration APPROVED by ADMINISTRATION',
       'Declaration APPROVED by BUDGET OWNER',
       'Declaration FOR_APPROVAL by SUPERVISOR',
       'Declaration REJECTED by ADMINISTRATION',
       'Declaration FOR_APPROVAL by PRE_APPROVER',
       'Declaration REJECTED by BUDGET OWNER',
       'Declaration FOR_APPROVAL by ADMINISTRATION'], dtype=object)

In [29]:
#logdata["lifecycle:transition"].unique()

In [32]:
#case:concept:name is caseid
logdata["case:concept:name"].unique()

array(['A1', 'A100', 'A10000', ..., 'V9997', 'V9998', 'V9999'],
      dtype=object)

In [30]:
#extract the columns, sort by time, convert activity to lower case
logdata.rename(columns={"concept:name":"activity", "time:timestamp":"timestamp","case:concept:name":"case_id"},inplace=True)
logdata = logdata.sort_values(by = ["timestamp","case_id"])
df = logdata[["timestamp","activity","case_id"]].copy()
df["activity"] = df["activity"].str.lower()
df.head()

,timestamp,activity,case_id
0,2017-01-09 09:49:50+00:00,declaration submitted by employee,declaration 86791
4,2017-01-09 10:26:14+00:00,declaration submitted by employee,declaration 86795
9,2017-01-09 11:13:33+00:00,declaration submitted by employee,declaration 86800
14,2017-01-09 11:24:20+00:00,declaration submitted by employee,declaration 86731
1,2017-01-09 11:27:48+00:00,declaration final_approved by supervisor,declaration 86791


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56437 entries, 0 to 52070
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype              
---  ------     --------------  -----              
 0   timestamp  56437 non-null  datetime64[ns, UTC]
 1   activity   56437 non-null  object             
 2   case_id    56437 non-null  object             
dtypes: datetime64[ns, UTC](1), object(2)
memory usage: 1.7+ MB


In [32]:
#Check if there are duplicates
print(df.duplicated().any())
print(df.duplicated().sum())

False
0


In [33]:
#Check activities for a case
df.drop_duplicates(inplace=True)
df.duplicated().sum()

0

In [34]:
df = df.reset_index(drop=True)
df.head()

,timestamp,activity,case_id
0,2017-01-09 09:49:50+00:00,declaration submitted by employee,declaration 86791
1,2017-01-09 10:26:14+00:00,declaration submitted by employee,declaration 86795
2,2017-01-09 11:13:33+00:00,declaration submitted by employee,declaration 86800
3,2017-01-09 11:24:20+00:00,declaration submitted by employee,declaration 86731
4,2017-01-09 11:27:48+00:00,declaration final_approved by supervisor,declaration 86791


In [37]:
#timestamps are not unique
df["timestamp"].duplicated().any()

True

In [38]:
df.to_csv("dmd_df.csv",index=False)